In [1]:
!pip install optuna

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from __future__ import division, print_function, unicode_literals,  absolute_import

%matplotlib inline

from google.colab import files

# Common imports
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import optuna
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, accuracy_score
import gc
from sklearn.preprocessing import StandardScaler

use_cuda = True
device = torch.device("cuda" if (use_cuda and torch.cuda.is_available()) else "cpu")

In [ ]:
device

In [ ]:
df = pd.read_csv("drive/MyDrive/data/train.csv")

df.head()

In [ ]:
print('shape',df.shape)
print('columns',df.columns.tolist())

# keep only numeric columns (daily percentage columns)
num = df.select_dtypes(include="number")

# flatten and count
positives = (num > 0).sum().sum()
negatives = (num < 0).sum().sum()
zeros = (num == 0).sum().sum()

print(f"positive values: {positives}")
print(f"negative values: {negatives}")
print(f"zeros: {zeros}")
print(f"total numeric entries: {num.size}")

In [ ]:
data_array = df.iloc[:, 1:].values
feature_dim =6

In [ ]:
class StockRNN(nn.Module):
    def __init__(self, num_stocks, embedding_dim, hidden_dim, seq_len, num_layers,
                 dropout, learning_rate, weight_decay):
        super(StockRNN, self).__init__()
        self.num_layers = num_layers
        self.hidden_size =  embedding_dim
        self.embedding = nn.Embedding(num_stocks, embedding_dim)
        self.rnn = nn.GRU(input_size=feature_dim + embedding_dim, hidden_size=hidden_dim,
                          num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate, weight_decay=weight_decay)
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer,
            T_0=5,
            T_mult=1,
            eta_min=5e-5)

    def forward(self, x_seq, stock_id):
        embbeding = self.embedding(stock_id)
        embbeding = embbeding.unsqueeze(1).repeat(1, x_seq.size(1), 1)
        x = torch.cat([x_seq, embbeding], dim=-1)
        rnn_out, _ = self.rnn(x)
        pooled = rnn_out.mean(dim=1)
        last_rnn_out = self.dropout(pooled)

        logits = self.fc(last_rnn_out).squeeze(1)
        return logits

    def train_one_epoch(self, dataloader):

        self.train()
        total_loss = 0
        for X, y, stocks in dataloader:
            X, stocks = X.to(device), stocks.to(device)
            y = y.to(device).float()
            logits = self(X, stocks)
            loss = self.criterion(logits, y)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.rnn.parameters(), 1.0)
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(dataloader)

    def predict(self, dataloader):
        self.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for X, y, stocks in dataloader:
                X, stocks = X.to(device), stocks.to(device)
                y = y.to(device).float()
                predicted = self(X, stocks)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
        return all_labels, all_preds

In [ ]:
class StockDataset(Dataset):
    def __init__(self, data_tensor, window_size=10):
        self.window_size = window_size
        self.data = data_tensor
        self.num_stocks = data_tensor.shape[0]
        self.total_days = data_tensor.shape[1]
        self.features = self.build_features(data_tensor)
        self.valid_samples_per_stock = self.total_days - window_size - 5

    def __len__(self):
        return self.num_stocks * self.valid_samples_per_stock

    def __getitem__(self, idx):
        stock_idx = idx // self.valid_samples_per_stock
        day_offset = idx % self.valid_samples_per_stock

        x_seq = self.features[
            stock_idx,
            day_offset : day_offset + self.window_size
        ]

        future_window = self.data[
            stock_idx,
            day_offset + self.window_size : day_offset + self.window_size + 5
        ]

        y = float(future_window.sum() > 0)
        return x_seq, y, torch.tensor(stock_idx, dtype=torch.long)

    def build_features(self, data):
        # data: [num_stocks, num_days]

        returns = data

        # rolling mean（vectorized）
        padded = F.pad(data.unsqueeze(1), (4, 0))  # 只在左边补

        mean_5 = F.avg_pool1d(
            padded, kernel_size=5, stride=1
        ).squeeze(1)

        # rolling std（近似写法）
        mean_sq = torch.nn.functional.avg_pool1d(
            (data**2).unsqueeze(1), kernel_size=5, stride=1, padding=2
        ).squeeze(1)

        padded_sq = F.pad((data**2).unsqueeze(1), (4, 0))

        mean_sq = F.avg_pool1d(
            padded_sq, kernel_size=5, stride=1
        ).squeeze(1)

        std_5 = torch.sqrt(mean_sq - mean_5**2 + 1e-6)

        # momentum
        momentum_5 = data - torch.roll(data, shifts=5, dims=1)
        momentum_5[:, :5] = 0

        # max/min（稍慢但只算一次）
        max_5 = data.clone()
        min_5 = data.clone()

        for i in range(data.shape[1]):
            max_5[:, i] = data[:, max(0, i-4):i+1].max(dim=1).values
            min_5[:, i] = data[:, max(0, i-4):i+1].min(dim=1).values

        features = torch.stack([
            returns,
            mean_5,
            std_5,
            momentum_5,
            max_5,
            min_5
        ], dim=-1)

        return features

In [ ]:
def find_best_threshold(probs, labels):
    best_acc = 0
    best_t = 0.5

    for t in np.linspace(0.4, 0.6, 41):
        preds = (probs > t).astype(int)
        acc = (preds == labels).mean()

        if acc > best_acc:
            best_acc = acc
            best_t = t

    return best_t, best_acc

In [ ]:
num_stocks, total_days = data_array.shape
split_idx = int(total_days * 0.9)

train_data_raw = data_array[:, :split_idx]
test_data_raw = data_array[:, split_idx:]


scaler = StandardScaler()
train_data_scaled = scaler.fit_transform(train_data_raw.T).T.astype(np.float32)
test_data_scaled = scaler.transform(test_data_raw.T).T.astype(np.float32)

train_tensor = torch.from_numpy(train_data_scaled)
test_tensor = torch.from_numpy(test_data_scaled).float()

print(f"Training Set has: {train_data_raw.shape[1]} days, Test Set has: {test_data_raw.shape[1]} days")

In [ ]:
def objective(trial):
    # hyperparameter search space
    lr = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [16, 32, 64])
    emb_dim = trial.suggest_categorical("emb_dim", [8, 16])
    win_size = trial.suggest_categorical("win_size", [20, 40])
    drop_rate = trial.suggest_float("dropout", 0.3, 0.6)
    num_layers = trial.suggest_categorical("num_layers", [1, 2])
    weight_decay = trial.suggest_categorical("weight_decay", [0, 1e-6, 1e-5, 1e-4])
    batch_size = 512

    # dataset prepare
    # subset_data = train_tensor[:40, :]
    train_ds = StockDataset(train_tensor, window_size=win_size)
    test_ds = StockDataset(test_tensor, window_size=win_size)

    train_loader = DataLoader(train_ds, batch_size=batch_size, pin_memory=True, num_workers=2)
    validation_loader = DataLoader(test_ds, batch_size=batch_size, pin_memory=True, num_workers=2)

    # model construct
    model = StockRNN(num_stocks=442, embedding_dim=emb_dim, hidden_dim=hidden_size, weight_decay=weight_decay,
                     seq_len=win_size, dropout=drop_rate, num_layers=num_layers, learning_rate=lr).to(device)

    best_acc = 0.0
    N_EPOCH = 15
    model.to(device)
    model.train()
    for epoch in range(N_EPOCH):
        train_avg_loss = model.train_one_epoch(train_loader)

        y_pred = []
        y_true = []
        val_loss = 0
        model.eval()
        with torch.no_grad():
            for X, y, stocks in validation_loader:
                X, stocks = X.to(device), stocks.to(device)
                y = y.to(device).float()
                logits = model(X, stocks)
                predicted = (logits > 0).float()

                val_loss += model.criterion(predicted, y).item()
                y_pred.extend(predicted.cpu().numpy())
                y_true.extend(y.cpu().numpy())
        val_avg_loss = val_loss / len(validation_loader)
        acc = accuracy_score(y_true, y_pred)

        if (epoch + 1) % 5 == 0:
           print(f"Val Avg Loss: {val_avg_loss:.4f} | Val Acc: {acc:.4f} | Train Avg Loss: {train_avg_loss:.4f}")

        trial.set_user_attr("accuracy", acc)
        trial.report(val_avg_loss, epoch)
        if trial.should_prune():
            print(f"Trial pruned at epoch {epoch}")
            raise optuna.exceptions.TrialPruned()

    del model, train_loader, validation_loader, train_ds, test_ds
    torch.cuda.empty_cache()
    gc.collect()
    return val_avg_loss


In [ ]:
db_path = "/content/drive/MyDrive/data/params_3_19.db"
study = optuna.create_study(
    direction="minimize",
    study_name="stock_prediction",
    storage= f"sqlite:///{db_path}",
    load_if_exists=True
)
study.optimize(objective, n_trials=5)

In [ ]:
study = optuna.load_study(
    study_name="stock_prediction",
    storage="sqlite:////content/drive/MyDrive/data/params_v2.db"
)
study.best_params


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0, path='checkpoint'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_acc = None
        self.best_loss = None
        self.early_stop = False
        self.delta = delta
        self.path = path

        if not os.path.exists(self.path):
            os.makedirs(self.path)

    def __call__(self, val_loss, val_acc, model):

        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_acc = val_acc
            self.save_checkpoint(val_loss, val_acc, model, "init")

        elif val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, val_acc, model, "best_loss")
            self.counter = 0 # 只要逻辑在优化，就给它机会继续跑

        elif val_acc > self.best_acc:
            self.best_acc = val_acc
            self.save_checkpoint(val_loss, val_acc, model, "best_acc")

        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, val_loss, val_acc, model, tag):
        '''
        tag: 'best_loss' 或 'best_acc'
        '''
        suffix = val_loss if tag == 'best_loss' else val_acc
        file_path = os.path.join(self.path, f"/content/drive/MyDrive/data/{tag}_model_{suffix}.pt")
        if self.verbose:
            print(f'[{tag.upper()}] New record! Loss: {val_loss:.4f}, Acc: {val_acc:.4f}. Saving...')

        torch.save(model.state_dict(), file_path)

In [ ]:
best_params = {
     'lr': 0.002060646413930728,
    'hidden_size': 128,
    'emb_dim': 8,
    'win_size': 40,
    'dropout': 0.16078598063917243,
    'num_layers': 1,
    'weight_decay': 0
}
EPOCHS = 20
PATIENCE = 8
######### Full Training ############
early_stopping = EarlyStopping(patience=PATIENCE, verbose=True)

train_ds = StockDataset(train_tensor, window_size=best_params['win_size'])
test_ds = StockDataset(test_tensor, window_size=best_params['win_size'])

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, pin_memory=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=512, pin_memory=True, num_workers=2)



model = StockRNN(num_stocks=442, embedding_dim=best_params['emb_dim'], hidden_dim=best_params['hidden_size'], weight_decay=best_params['weight_decay'],
                 seq_len=best_params['win_size'], dropout=best_params["dropout"], learning_rate=best_params['lr'],
                 num_layers=best_params['num_layers']).to(device)

checkpoint = torch.load('/content/drive/MyDrive/data/best_loss_model_3_19.pt')
model.load_state_dict(checkpoint)

best_acc = 0.0
model.to(device)
for epoch in range(EPOCHS):
    model.train()
    avg_loss = model.train_one_epoch(train_loader)

    y_pred = []
    y_true = []
    probs = []
    best_val_loss = 0
    val_loss = 0
    model.eval()
    with torch.no_grad():
        for X, y, stocks in test_loader:
            X, stocks = X.to(device), stocks.to(device)
            y = y.to(device).float()
            logits = model(X, stocks)
            predicted = (logits > 0).float()
            val_loss += model.criterion(predicted, y).item()

            # probs.append(torch.sigmoid(logits).cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(y.cpu().numpy())

    val_avg_loss = val_loss / len(test_loader)
    acc = accuracy_score(y_true, y_pred)
    # best_t, best_acc = find_best_threshold(probs, y_true)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_loss:.4f} | Val Loss: {val_avg_loss:.4f} | Val Acc: {acc:.4f} | Current LR: {current_lr:.6f}")

    if acc > best_acc:
        best_acc = acc

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'model_lowest_loss.pt')

    model.scheduler.step()
    current_lr = model.optimizer.param_groups[0]['lr']

    early_stopping(val_avg_loss, acc, model)

    if early_stopping.early_stop:
        print("Early stopping triggered. Training stopped.")
        break


del model, train_loader, test_loader, train_ds, test_ds
torch.cuda.empty_cache()
gc.collect()

print(f"Best Val Acc: {best_acc:.4f}")

In [ ]:
import gc
import torch

# 1. 清理 Python 垃圾回收
gc.collect()

# 2. 清理显存（非常重要！）
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect() # 清理多进程留下的缓存

In [ ]:
#################Prediction#############################
final_model = StockRNN(num_stocks=442, embedding_dim=best_params['emb_dim'], hidden_dim=best_params['hidden_size'],
                       seq_len=best_params['win_size'], dropout=best_params["dropout"], learning_rate=best_params['lr']).to(device)
final_model.load_state_dict(torch.load('/content/drive/MyDrive/data/best_loss_model.pt'))
final_model.eval()

scaler = StandardScaler()
full_data_scaled = scaler.fit_transform(data_array.T).T
full_data_tensor =  torch.from_numpy(full_data_scaled).float()
win_size = best_params["win_size"]
all_preds = []

print("Predicting 2022/04/01 daily change for all stocks ...")
all_logits_list = []
final_preds = []
for i in range(442):
    raw_window = data_array[i, -win_size:]
    window_mean = raw_window.mean()
    window_std = raw_window.std() + 1e-8
    norm_window = (raw_window - window_mean) / window_std

    last_window = torch.from_numpy(norm_window).float().unsqueeze(0).unsqueeze(-1).to(device)
    stock_idx = torch.tensor([i]).to(device)

    with torch.no_grad():
        logits = final_model(last_window, stock_idx)
        all_logits_list.append(logits)
        # turning logits to label
        predicted = (logits > 0).float()
        final_preds.append(predicted)

all_logits_list_np = np.array(all_logits_list)
threshold = np.median(all_logits_list)

print(f"Logits Maximum: {all_logits_list_np.max():.4f}")
print(f"Logits Minimum: {all_logits_list_np.min():.4f}")
print(f"Threshold: {threshold:.4f}")


from collections import Counter
print("Distribution After rebalanced:", Counter(final_preds))

submission = pd.DataFrame({
    'ID': [f'company_{i}' for i in range(442)],
    'value': final_preds
})

submission.to_csv('final_submission.csv', index=False)
print("submission.csv generated")

print("\prediction distribution：")
print(submission['value'].value_counts())